# PyTorch Tutorial — From Zero to CNN

This notebook teaches PyTorch one concept at a time, building up to the CNN image classifier you've already seen.

**Roadmap:**
1. Tensors — PyTorch's core data structure
2. Autograd — how PyTorch learns (automatic differentiation)
3. `nn.Module` — how to define a model
4. Loss functions and optimizers
5. The training loop
6. Putting it all together — the CNN

Work through each section, run the cells, and read the explanations before moving on.

---
# Concept 1 — Tensors

A **tensor** is the fundamental data structure in PyTorch. Think of it as a generalisation of arrays:

| Name | Shape example | What it looks like |
|---|---|---|
| Scalar | `()` | A single number: `3.14` |
| Vector | `(4,)` | A 1D list: `[1, 2, 3, 4]` |
| Matrix | `(3, 4)` | A 2D table / spreadsheet |
| 3D tensor | `(3, 32, 32)` | A colour image (3 channels, 32×32 pixels) |
| 4D tensor | `(32, 3, 32, 32)` | A **batch** of 32 colour images |

In the CNN notebook, every image becomes a `(3, 32, 32)` tensor, and a batch becomes `(32, 3, 32, 32)`.

In [ ]:
import torch

# --- Creating tensors ---
scalar = torch.tensor(3.14)
vector = torch.tensor([1.0, 2.0, 3.0, 4.0])
matrix = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])

print("Scalar:", scalar, "| shape:", scalar.shape)
print("Vector:", vector, "| shape:", vector.shape)
print("Matrix:\n", matrix, "| shape:", matrix.shape)

In [ ]:
import matplotlib.pyplot as plt

# --- Useful ways to create tensors ---
zeros  = torch.zeros(3, 4)          # all zeros, shape (3,4)
ones   = torch.ones(2, 3)           # all ones
rand   = torch.rand(3, 32, 32)      # random values in [0,1] — like a fake image!
arange = torch.arange(0, 10, 2)     # like Python range()

print("zeros shape:", zeros.shape)
print("rand shape (fake image):", rand.shape)
print("arange:", arange)

# --- Display the fake image ---
# rand is (3, 32, 32) — channels first (PyTorch format)
# matplotlib expects (32, 32, 3) — channels last, so we permute
img = rand.permute(1, 2, 0).numpy()   # (C, H, W) → (H, W, C)

plt.figure(figsize=(3, 3))
plt.imshow(img)
plt.title("Fake image: torch.rand(3, 32, 32)")
plt.axis('off')
plt.show()

In [ ]:
# --- Tensor operations feel just like NumPy ---
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("Add:     ", a + b)
print("Multiply:", a * b)          # element-wise
print("Dot product:", torch.dot(a, b))  # 1*4 + 2*5 + 3*6 = 32
print("Mean:", a.mean())
print("Sum: ", a.sum())

In [ ]:
# --- Shape manipulation — you'll see this constantly ---
x = torch.rand(4, 8)      # 4 rows, 8 columns
print("Original shape:", x.shape)

# reshape: same data, different shape
x_reshaped = x.reshape(2, 16)
print("Reshaped:", x_reshaped.shape)

# flatten: collapse all dims into one (used just before fully connected layers in CNN)
x_flat = x.flatten()
print("Flattened:", x_flat.shape)

# unsqueeze: add a dimension (e.g. single image → batch of 1)
img = torch.rand(3, 32, 32)     # one image
batch = img.unsqueeze(0)         # add batch dimension at position 0
print("Image:", img.shape, "→ Batch:", batch.shape)

In [ ]:
# --- unsqueeze: adding a dimension ---
# Think of it as wrapping the tensor in an extra layer of brackets

x = torch.tensor([10.0, 20.0, 30.0])   # shape: (3,)
print("Original:        ", x.shape, "→", x.tolist())

# unsqueeze(0) — add a new dimension at position 0 (outermost)
a = x.unsqueeze(0)
print("unsqueeze(0):    ", a.shape, "→", a.tolist())   # [[10, 20, 30]]

# unsqueeze(1) — add a new dimension at position 1 (innermost)
b = x.unsqueeze(1)
print("unsqueeze(1):    ", b.shape, "→", b.tolist())   # [[10], [20], [30]]

# --- Why does this matter? ---
# Most PyTorch models expect a BATCH as input, not a single sample.
# If your model expects shape (batch, features), a single sample (3,) won't work.
# unsqueeze(0) adds the batch dimension: (3,) → (1, 3) = batch of 1

single = torch.tensor([1.0, 2.0, 3.0])     # shape (3,) — one sample
batch  = single.unsqueeze(0)                 # shape (1, 3) — batch of 1 sample
print("\nSingle sample:", single.shape)
print("As a batch:   ", batch.shape)

### Checkpoint — what you just learned
- A tensor is an N-dimensional array
- Images in PyTorch are `(channels, height, width)` tensors
- A batch of images is `(batch_size, channels, height, width)`
- `shape`, `reshape`, `flatten`, `unsqueeze` are the most common operations you'll see

**In the CNN notebook:** `images, labels = next(iter(trainloader))` gives you a tensor of shape `(32, 3, 32, 32)` — 32 images, 3 colour channels, 32×32 pixels.

---
# Concept 2 — Autograd (How PyTorch Learns)

This is the magic behind all neural network training.

**The idea:** When you do maths with tensors, PyTorch secretly records every operation in a **computation graph**. When you call `.backward()`, it walks back through that graph and computes the **gradient** (derivative) of the loss with respect to every weight.

A gradient tells you: *"if I increase this weight slightly, does the loss go up or down, and by how much?"*  
The optimizer then nudges each weight in the direction that reduces the loss.

You only need to understand three things:
- `requires_grad=True` — tell PyTorch to track this tensor
- `loss.backward()` — compute all gradients
- `tensor.grad` — read the gradient

In [ ]:
import torch

# --- Simple example: y = x^2, what is dy/dx at x=3? ---
# Answer should be 2x = 6

x = torch.tensor(3.0, requires_grad=True)   # track this value
y = x ** 2                                   # y = x^2

y.backward()   # compute gradient
print("dy/dx at x=3:", x.grad)              # should print 6.0

In [ ]:
# --- More realistic: a tiny 1-neuron network ---
# We have: output = weight * input + bias
# We want to know: how should we adjust weight and bias to reduce error?

weight = torch.tensor(2.0, requires_grad=True)
bias   = torch.tensor(1.0, requires_grad=True)
x      = torch.tensor(3.0)          # input (no grad needed)

# Forward pass
output = weight * x + bias           # 2*3 + 1 = 7
target = torch.tensor(10.0)          # what we wanted
loss   = (output - target) ** 2      # squared error: (7-10)^2 = 9

print("Output:", output.item())
print("Loss:  ", loss.item())

# Backward pass — computes gradients
loss.backward()

print("d(loss)/d(weight):", weight.grad)   # how much loss changes if weight increases
print("d(loss)/d(bias):  ", bias.grad)     # how much loss changes if bias increases

In [ ]:
# --- torch.no_grad() --- 
# During evaluation/testing we DON'T want to track gradients — it wastes memory.
# This is why the CNN notebook wraps the test loop in torch.no_grad()

weight = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():                   # gradients NOT tracked inside here
    output = weight * 5

print("Has grad_fn?", output.grad_fn)   # None — not tracked
print("Value:", output.item())

### Checkpoint — what you just learned
- PyTorch tracks every computation on `requires_grad=True` tensors
- `.backward()` computes all gradients automatically — you never do calculus by hand
- Gradients tell the optimizer which direction to nudge each weight
- `torch.no_grad()` disables tracking for efficiency — used during testing

**In the CNN notebook:**
```python
loss.backward()    # compute gradients for all model weights
optimizer.step()   # use those gradients to update weights
```
with `torch.no_grad():` wrapping the entire test loop.

---
# Concept 3 — `nn.Module` (Defining a Model)

All PyTorch models are Python classes that inherit from `nn.Module`. You define:
- `__init__`: declare your layers
- `forward`: describe how data flows through them

PyTorch handles everything else — parameter tracking, gradients, moving to GPU, saving/loading.

Let's build up from a single neuron to the full CNN.

- This operation is a matrix multiplication + bias addition:
- output = x @ W.T + b

  Concretely with nn.Linear(4, 2) — 4 inputs, 2 outputs:

  - W is a weight matrix of shape (2, 4) — 2 output neurons, each has 4 weights
  - b is a bias vector of shape (2,)

  Written out fully:

  x = [1.0,  2.0,  3.0,  4.0]

  W = [[w11, w12, w13, w14],    ← weights for output neuron 1
       [w21, w22, w23, w24]]    ← weights for output neuron 2

  b = [b1, b2]

  Each output neuron computes a weighted sum of all inputs:

  out[0] = (1.0×w11) + (2.0×w12) + (3.0×w13) + (4.0×w14) + b1
  out[1] = (1.0×w21) + (2.0×w22) + (3.0×w23) + (4.0×w24) + b2

  You can verify this yourself:

  layer = nn.Linear(4, 2)

  - inspect the randomly initialised weights
  print("W shape:", layer.weight.shape)   # (2, 4)
  print("b shape:", layer.bias.shape)     # (2,)

  x = torch.tensor([1.0, 2.0, 3.0, 4.0])

 - manual calculation
  manual = x @ layer.weight.T + layer.bias

- via layer
  auto = layer(x)

  print("Manual:", manual)
  print("Auto:  ", auto)    # identical

  ---
  Why is this useful?
  
  Each output neuron learns its own set of weights — what to pay attention to in the input. After training, out[0] might represent "does this look like a cat?" and out[1]
  "does this look like a dog?" — purely by adjusting those w values through backpropagation.

In [ ]:
import torch
import torch.nn as nn

# --- A single Linear (fully connected) layer ---
# nn.Linear(in_features, out_features) applies: output = input @ weight.T + bias

layer = nn.Linear(4, 2)      # takes 4 inputs, produces 2 outputs
print("W shape:", layer.weight.shape)   # (2, 4)
print("b shape:", layer.bias.shape)     # (2,)

x = torch.tensor([1.0, 2.0, 3.0, 4.0])
out = layer(x)
print("Input shape:", x.shape)
print("Output shape:", out.shape)
print("Output:", out)   # two numbers — one per output neuron

manual = x @ layer.weight.T + layer.bias
print(manual)
print(out) #auto

W shape: torch.Size([2, 4])
b shape: torch.Size([2])
Input shape: torch.Size([4])
Output shape: torch.Size([2])
Output: tensor([-3.1425,  0.2067], grad_fn=<ViewBackward0>)


---
# Minimal Training Loop — 4-input state, 2 actions (RL flavour)

We have a game state with 4 features. The model must learn to predict which action (0 or 1) is correct.
No dataset, no DataLoader — just raw tensors so every step is visible.

In [ ]:
import torch
import torch.nn as nn

# --- Input: 4 game states, each with 4 features ---
# Think: [position, velocity, angle, score]
X = torch.tensor([
    [1.0, 0.0, 0.5, 0.2],   # state 0 → correct action: 0
    [0.0, 1.0, 0.3, 0.8],   # state 1 → correct action: 1
    [0.9, 0.1, 0.6, 0.1],   # state 2 → correct action: 0
    [0.2, 0.8, 0.1, 0.9],   # state 3 → correct action: 1
])
y = torch.tensor([0, 1, 0, 1])   # correct actions (labels)

# --- Model: 4 inputs → 8 hidden neurons → 2 action scores ---
model = nn.Sequential(
    nn.Linear(4, 8),    # layer 1: each of 8 neurons listens to all 4 inputs
    nn.ReLU(),          # non-linearity
    nn.Linear(8, 2)     # layer 2: 2 neurons → score for action 0, score for action 1
)

criterion = nn.CrossEntropyLoss()                        # measures how wrong we are
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- Training loop ---
for epoch in range(200):

    output = model(X)                   # Step 1: forward — shape (4, 2)
    loss   = criterion(output, y)       # Step 2: how wrong?

    optimizer.zero_grad()               # Step 3: clear old gradients
    loss.backward()                     # Step 4: who caused the error?
    optimizer.step()                    # Step 5: nudge weights

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

# --- What did the model learn? ---
model.eval()
with torch.no_grad():
    scores  = model(X)                  # shape (4, 2) — raw scores per action
    actions = scores.argmax(dim=1)      # pick action with highest score

print("\nPredicted actions:", actions.tolist())
print("True actions:     ", y.tolist())
print("Scores:\n", scores.round(decimals=2))

---
# From Linear to CNN — Same Loop, Different Model

The training loop you just ran never changes.  
Swapping `nn.Linear` for `nn.Conv2d` is the only difference.

In [ ]:
import torch
import torch.nn as nn

# --- Why Linear fails for images ---
# A tiny 8x8 grayscale image = 64 numbers
# A real image 84x84 = 7,056 numbers — Linear connects EVERY pixel to EVERY neuron

image_flat = torch.rand(64)       # 8x8 image flattened to 64 numbers

linear_model = nn.Linear(64, 2)   # every pixel → every neuron
print("Linear parameters:", sum(p.numel() for p in linear_model.parameters()))
# 64×2 weights + 2 biases = 130 params (manageable for 8x8, but explodes for real images)

# For 84x84: nn.Linear(7056, 128) = 903,296 params in ONE layer alone
print("Linear params for 84x84 → 128:", 84*84*128 + 128)

In [ ]:
import torch
import torch.nn as nn

# --- Conv2d: small filter slides over the whole image ---
# Much fewer parameters, respects spatial structure

conv_model = nn.Sequential(
    nn.Conv2d(1, 4, kernel_size=3),  # 1 channel in, 4 filters, 3×3 kernel
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(4 * 6 * 6, 2)         # 4 filters × 6×6 spatial = 144 → 2 actions
)

# Shape flow — run a fake 8x8 grayscale image through
x = torch.rand(1, 1, 8, 8)          # (batch=1, channels=1, H=8, W=8)
print("Input shape:        ", x.shape)

after_conv = nn.Conv2d(1, 4, kernel_size=3)(x)
print("After Conv2d:       ", after_conv.shape)   # (1, 4, 6, 6) — 8-3+1=6

after_flat = after_conv.flatten(start_dim=1)
print("After Flatten:      ", after_flat.shape)   # (1, 144)

out = conv_model(x)
print("Final output:       ", out.shape)           # (1, 2) — 2 action scores

print("\nConv2d parameters: ", sum(p.numel() for p in conv_model.parameters()))
# 3×3 kernel × 4 filters + biases = far fewer than Linear on raw pixels

In [ ]:
import torch
import torch.nn as nn

# --- Same 5-step training loop, CNN model ---
# 4 fake 8x8 grayscale "game screens", 2 possible actions

X = torch.rand(4, 1, 8, 8)          # (batch=4, channels=1, H=8, W=8)
y = torch.tensor([0, 1, 0, 1])      # correct actions

model = nn.Sequential(
    nn.Conv2d(1, 4, kernel_size=3),
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(4 * 6 * 6, 2)
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    output = model(X)                    # Step 1: forward — (4, 2)
    loss   = criterion(output, y)        # Step 2: how wrong?

    optimizer.zero_grad()                # Step 3: clear gradients
    loss.backward()                      # Step 4: backprop
    optimizer.step()                     # Step 5: update weights

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    actions = model(X).argmax(dim=1)

print("\nPredicted actions:", actions.tolist())
print("True actions:     ", y.tolist())
# Notice: identical loop to the Linear version — only the model definition changed

In [ ]:
# --- Activation functions ---
# Without activations, stacking linear layers is pointless (still just linear)
# Activations add non-linearity, letting the network learn complex patterns

import torch.nn.functional as F

x = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])

print("Input:       ", x)
print("ReLU:        ", F.relu(x))       # max(0, x) — kills negatives
print("Sigmoid:     ", torch.sigmoid(x)) # squishes to (0,1)
print("Tanh:        ", torch.tanh(x))    # squishes to (-1,1)

In [ ]:
# --- A minimal model using nn.Module ---
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)    # layer 1: 4 inputs → 8 hidden neurons
        self.fc2 = nn.Linear(8, 3)    # layer 2: 8 hidden → 3 outputs (3 classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))   # pass through layer 1, then ReLU
        x = self.fc2(x)            # pass through layer 2 (no activation on output)
        return x

model = TinyNet()
print(model)

x = torch.rand(4)             # one sample with 4 features
out = model(x)                # calling the model runs forward()
print("\nInput shape:", x.shape)
print("Output shape:", out.shape)   # 3 scores, one per class

In [ ]:
# --- nn.Sequential: a cleaner way to stack layers ---
# Instead of writing forward() manually, just list the layers in order
# This is exactly how SimpleCNN is written in the CNN notebook!

model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)

x = torch.rand(4)
out = model(x)
print("Output:", out)    # same result as TinyNet above

In [ ]:
# --- Introducing Conv2d ---
# The building block of image models.
# Instead of connecting every input to every output (like Linear),
# it slides a small filter (kernel) across the image and detects local patterns.

# nn.Conv2d(in_channels, out_channels, kernel_size, padding)
# in_channels:  how many colour channels the input has (3 for RGB)
# out_channels: how many different filters to learn
# kernel_size:  size of each filter (3 means a 3x3 filter)
# padding=1:    pad the image border so output size stays the same

conv = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)

# Fake batch: 2 images, 3 channels, 32x32 pixels
fake_batch = torch.rand(2, 3, 32, 32)
output = conv(fake_batch)

print("Input shape: ", fake_batch.shape)  # (batch, channels, H, W)
print("Output shape:", output.shape)      # (batch, 32 filters, H, W)

In [ ]:
# --- MaxPool2d ---
# Shrinks the spatial dimensions by keeping only the max value in each window.
# MaxPool2d(2, 2) halves the height and width.
# Purpose: reduce computation, and make features location-invariant.

pool = nn.MaxPool2d(2, 2)

x = torch.rand(1, 32, 32, 32)   # 1 image, 32 channels, 32x32
out = pool(x)
print("Before pool:", x.shape)    # 32x32
print("After pool: ", out.shape)  # 16x16 — halved!

### Checkpoint — what you just learned
- `nn.Module` is the base class for all PyTorch models
- `nn.Linear` is a fully connected layer
- `nn.Conv2d` slides filters across an image to detect patterns
- `nn.MaxPool2d` halves the image size, keeping the strongest signals
- `nn.Sequential` lets you stack layers without writing `forward()` manually
- Activation functions (ReLU) add non-linearity between layers

**In the CNN notebook:** `SimpleCNN` uses exactly Conv2d → ReLU → MaxPool repeated 3 times, then Flatten → Linear → Linear — all inside `nn.Sequential`.

---
# Concept 4 — Loss Functions and Optimizers

Once you have a model, you need two more things to train it:

**Loss function** — measures how wrong the model is. Training goal: minimise this number.

**Optimizer** — uses the gradients (from `.backward()`) to update the model weights.

The loop is always:  
`forward → compute loss → backward → optimizer step`

In [ ]:
import torch
import torch.nn as nn

# --- CrossEntropyLoss ---
# Used for multi-class classification (exactly what CIFAR-10 is)
# Input: raw scores (logits) for each class — NOT probabilities
# Target: the correct class index as an integer

criterion = nn.CrossEntropyLoss()

# Say we have 3 images, 4 possible classes
# Model outputs raw scores (logits) — higher = more confident
predictions = torch.tensor([
    [2.0, 1.0, 0.1, 0.5],   # image 1: model thinks class 0
    [0.1, 3.0, 0.2, 0.1],   # image 2: model thinks class 1
    [0.5, 0.2, 0.1, 4.0],   # image 3: model thinks class 3
])
labels = torch.tensor([0, 1, 2])   # true classes: 0, 1, 2 (last one is WRONG)

loss = criterion(predictions, labels)
print("Loss:", loss.item())   # higher because image 3 was misclassified

In [ ]:
# --- Loss intuition: perfect vs bad predictions ---

# Perfect predictions (correct class has very high score)
perfect = torch.tensor([[10.0, -1.0, -1.0]])
label   = torch.tensor([0])   # class 0 is correct
print("Perfect prediction loss:", criterion(perfect, label).item())   # very small

# Bad predictions (correct class has very low score)
bad = torch.tensor([[-10.0, 5.0, 5.0]])
print("Bad prediction loss:    ", criterion(bad, label).item())       # very large

In [ ]:
# --- Optimizers ---
# They read the .grad on each parameter and update the value to reduce loss.
# Adam is the most popular: it adapts the learning rate per parameter automatically.

import torch.nn as nn

model = nn.Linear(3, 1)    # tiny model as an example

# Adam optimizer — give it the model parameters and a learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# --- The four lines at the heart of every training loop ---
x = torch.rand(3)
target = torch.tensor([1.0])

pred = model(x)                        # 1. forward pass
loss = nn.MSELoss()(pred, target)       # 2. compute loss
optimizer.zero_grad()                   # 3a. clear old gradients (IMPORTANT!)
loss.backward()                         # 3b. compute new gradients
optimizer.step()                        # 4. update weights

print("Loss after one step:", loss.item())
print("(run multiple times to see loss decrease)")

### Why `optimizer.zero_grad()`?

PyTorch **accumulates** gradients by default. If you don't clear them, the gradient from step 2 gets added to step 1's gradient, and your updates become nonsense.

The sequence is always:
```
optimizer.zero_grad()   ← clear old gradients
loss.backward()         ← compute fresh gradients
optimizer.step()        ← apply them
```

In the CNN notebook, `zero_grad()` is called at the start of every batch, so gradients never bleed between batches.

---
# Concept 5 — The Training Loop

Now we combine everything into a real training loop.  
Let's train a model on a simple toy problem first, then you'll recognise exactly the same pattern in the CNN.

**Vocabulary:**
- **Epoch**: one full pass through the entire training dataset
- **Batch**: a small chunk of the dataset processed at once (e.g. 32 images)
- **Iteration**: one forward+backward pass on one batch
- **Steps per epoch**: `total_samples / batch_size` (e.g. 50000/32 ≈ 1563 for CIFAR-10)

In [ ]:
import torch
import torch.nn as nn

# --- Toy problem: learn to classify XOR ---
# Input: 2 bits. Output: their XOR (0 or 1)
# A linear model can't solve XOR — we need a hidden layer!

X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y = torch.tensor([0, 1, 1, 0])   # XOR labels

# Model
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 2)    # 2 output scores: class 0 and class 1
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- Training loop ---
for epoch in range(200):
    model.train()                    # set to training mode

    preds = model(X)                 # forward pass
    loss  = criterion(preds, y)      # compute loss

    optimizer.zero_grad()            # clear old gradients
    loss.backward()                  # backprop
    optimizer.step()                 # update weights

    if (epoch+1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

# --- Check predictions ---
model.eval()
with torch.no_grad():
    out = model(X).argmax(dim=1)    # pick the class with highest score
print("\nPredictions:", out.tolist())
print("True labels: ", y.tolist())

In [ ]:
# --- DataLoader: the standard way to feed data in batches ---
# In the CNN notebook, trainloader does exactly this for CIFAR-10

from torch.utils.data import TensorDataset, DataLoader
import torch
import torch.nn as nn

# Fake dataset: 100 samples, 4 features each, 3 classes
X = torch.rand(100, 4)
y = torch.randint(0, 3, (100,))

dataset = TensorDataset(X, y)
loader  = DataLoader(dataset, batch_size=16, shuffle=True)

model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# --- Training loop with batches ---
for epoch in range(3):
    total_loss = 0.0

    for batch_X, batch_y in loader:     # iterate over batches
        preds = model(batch_X)
        loss  = criterion(preds, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")

### Checkpoint — what you just learned
This pattern:
```python
for epoch in range(num_epochs):
    model.train()
    for images, labels in trainloader:
        predictions = model(images)
        loss = criterion(predictions, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
```
is **exactly** the training loop in the CNN notebook — just applied to images instead of random data.

- `model.train()` enables dropout/batchnorm training behaviour (good habit even without them)
- `model.eval()` + `torch.no_grad()` is used for the test loop

---
# Concept 6 — Putting It All Together: Annotated CNN

Now let's look at the full CNN notebook code, but heavily annotated so every line makes sense.  
Run this section after installing torchvision:
```
conda install torchvision -c pytorch
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# --- Hyperparameters ---
BATCH_SIZE   = 32      # process 32 images at a time
LEARNING_RATE = 0.001  # how big each weight update step is
NUM_EPOCHS   = 5       # how many times to loop over all training data

In [ ]:
# --- Data preprocessing ---
# transforms.Compose: chain multiple transforms together
# ToTensor:   converts PIL image (H x W x C, values 0-255) → tensor (C x H x W, values 0-1)
# Normalize:  (value - mean) / std  →  shifts values from [0,1] to roughly [-1, 1]
#             helps the optimizer converge faster (centred, consistent scale)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],   # one mean per channel (R, G, B)
                         std =[0.5, 0.5, 0.5])
])

# Download CIFAR-10 and wrap in DataLoaders
# DataLoader handles shuffling, batching, and parallel loading (num_workers)
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training images: {len(trainset):,}")
print(f"Test images:     {len(testset):,}")
print(f"Batches per epoch: {len(trainloader)}")

In [ ]:
# --- What does one batch look like? ---
images, labels = next(iter(trainloader))   # grab one batch

print("Batch tensor shape:", images.shape)   # (32, 3, 32, 32)
print("Labels shape:      ", labels.shape)   # (32,) — one class index per image
print("First few labels:  ", labels[:8].tolist())    # e.g. [3, 8, 8, 0, 6, 6, 1, 6]

classes = trainset.classes
print("Class names:", classes)

In [ ]:
# --- CNN Architecture ---
# Three stages of: Conv (detect features) → ReLU (non-linearity) → MaxPool (shrink)
# Then flatten and classify with fully connected layers
#
# Spatial dimensions after each stage:
#   Input:     (batch, 3,   32, 32)  ← 3-channel 32x32 image
#   After pool1: (batch, 32,  16, 16)  ← 32 feature maps, halved
#   After pool2: (batch, 64,   8,  8)  ← 64 feature maps, halved again
#   After pool3: (batch, 128,  4,  4)  ← 128 feature maps, halved again
#   After flatten: (batch, 128*4*4) = (batch, 2048)
#   After fc1:   (batch, 512)
#   After fc2:   (batch, 10)            ← 10 class scores

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            # --- Stage 1 ---
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 3 channels in → 32 feature maps
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 32x32 → 16x16

            # --- Stage 2 ---
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 32 → 64 feature maps
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 16x16 → 8x8

            # --- Stage 3 ---
            nn.Conv2d(64, 128, kernel_size=3, padding=1),# 64 → 128 feature maps
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 8x8 → 4x4

            # --- Classifier ---
            nn.Flatten(),                                 # (128, 4, 4) → 2048
            nn.Linear(128 * 4 * 4, 512),                 # 2048 → 512
            nn.ReLU(),
            nn.Linear(512, 10)                            # 512 → 10 class scores
        )

    def forward(self, x):
        return self.model(x)   # just pass through Sequential

# Pick GPU if available, otherwise CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = SimpleCNN().to(device)   # move all model parameters to device
print(model)

In [ ]:
# --- Loss and optimizer ---
criterion = nn.CrossEntropyLoss()                          # for multi-class classification
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --- Training loop ---
for epoch in range(NUM_EPOCHS):
    model.train()          # training mode (enables dropout, batchnorm updates if used)
    running_loss = 0.0

    for images, labels in trainloader:
        # Move data to the same device as the model
        images = images.to(device)   # shape: (32, 3, 32, 32)
        labels = labels.to(device)   # shape: (32,)

        # Forward pass: get 10 scores for each image
        predictions = model(images)                   # shape: (32, 10)

        # Compute how wrong we are
        loss = criterion(predictions, labels)

        # Backward pass: compute gradients and update weights
        optimizer.zero_grad()    # clear gradients from last step
        loss.backward()          # compute new gradients
        optimizer.step()         # apply gradients

        running_loss += loss.item()

    avg_loss = running_loss / len(trainloader)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Loss: {avg_loss:.4f}")

In [ ]:
# --- Evaluation ---
model.eval()                  # evaluation mode
correct = total = 0

with torch.no_grad():         # no gradients needed — saves memory and computation
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)                  # (batch, 10) raw scores

        # torch.max returns (max_value, index_of_max)
        # dim=1 means: find max across the 10 class scores
        _, predicted = torch.max(outputs, dim=1) # predicted shape: (batch,)

        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%  ({correct}/{total} correct)")

In [ ]:
# --- Visualise predictions ---
model.eval()
images, labels = next(iter(testloader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    preds = model(images).argmax(dim=1)   # class with highest score

fig, axes = plt.subplots(1, 8, figsize=(16, 3))
for i, ax in enumerate(axes):
    # Unnormalize: reverse the (x - 0.5) / 0.5 transform back to [0,1] for display
    img = images[i].cpu().permute(1, 2, 0).numpy() / 2 + 0.5
    ax.imshow(img)
    ax.axis('off')
    true_name = testset.classes[labels[i]]
    pred_name = testset.classes[preds[i]]
    color = 'green' if true_name == pred_name else 'red'
    ax.set_title(f"T:{true_name}\nP:{pred_name}", color=color, fontsize=8)

plt.tight_layout()
plt.show()

---
# Summary — What You Learned

| Concept | What it is | Where it appears in CNN notebook |
|---|---|---|
| **Tensor** | N-dimensional array | `images` shape `(32,3,32,32)` |
| **Autograd** | Auto-computes gradients | `loss.backward()` |
| **nn.Module** | Base class for all models | `class SimpleCNN(nn.Module)` |
| **Conv2d** | Slide filters over image | First 3 layers of SimpleCNN |
| **MaxPool2d** | Halve spatial dimensions | After each Conv layer |
| **CrossEntropyLoss** | Measures classification error | `criterion = nn.CrossEntropyLoss()` |
| **Adam** | Updates weights using gradients | `optimizer = torch.optim.Adam(...)` |
| **Training loop** | Forward → loss → backward → step | The `for epoch / for batch` loop |
| **DataLoader** | Feed data in shuffled batches | `trainloader`, `testloader` |
| **model.eval() + no_grad** | Disable training behaviour | The test loop |

### What to try next
- Add a **Dropout** layer (`nn.Dropout(0.3)`) before the classifier to reduce overfitting
- Add **Batch Normalisation** (`nn.BatchNorm2d(32)`) after each Conv layer to train faster
- Increase to 10 epochs and watch accuracy improve
- Add **data augmentation** (random flips/crops in the transform) for better generalisation